In [17]:
# ============================================================
# 🔌 LADEINFRASTRUKTUR + STRASSENNETZ (FINAL CLEAN VERSION)
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx

# ============================================================
# 1. GRAPH LADEN
# ============================================================

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_geo_com.pkl", "rb") as f:
    berlin_graph = pickle.load(f)

print("Keys:", berlin_graph.keys())

# arcs = edges
df_edges = pd.DataFrame(berlin_graph["arcs"])

# node coords direkt nutzen
node_coords = berlin_graph["node_coords"]

node_ids = np.array(list(node_coords.keys()))
node_lats = np.array([node_coords[n][0] for n in node_ids])
node_lons = np.array([node_coords[n][1] for n in node_ids])

print(f"Knoten: {len(node_ids):,}")
print(f"Kanten: {len(df_edges):,}")

# ============================================================
# 2. DISTANZEN
# ============================================================

Dist_dict = berlin_graph["Dist"]

df_edges["distance"] = [
    Dist_dict.get((row["u"], row["v"]), 0)
    for _, row in df_edges.iterrows()
]

# ============================================================
# 3. NEAREST NODE
# ============================================================

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

def nearest_node(lat, lon):
    dists = haversine_vectorized(lat, lon, node_lats, node_lons)
    idx = np.argmin(dists)
    return int(node_ids[idx]), float(dists[idx])

# ============================================================
# 4. CSV LADEN (FINAL)
# ============================================================

file_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Ladesaeulenregister_BNetzA_2026-04-22.csv"

charging_df = pd.read_csv(
    file_path,
    sep=";",
    encoding="latin-1",
    header=10
)

# Spalten trimmen (wichtiger Safety Schritt)
charging_df.columns = charging_df.columns.str.strip()

print("\nSpalten gefunden:")
print(charging_df.columns.tolist())

# relevante Spalten auswählen
charging_df = charging_df[[
    "Ladeeinrichtungs-ID",
    "Breitengrad",
    "Längengrad",
    "Nennleistung Ladeeinrichtung [kW]"
]].dropna()

charging_df.columns = ["id", "lat", "lon", "power_kw"]

print(f"Ladesäulen geladen: {len(charging_df)}")

# 🔥 ganz wichtig: zu float konvertieren (deutsche Zahlen fixen)

charging_df["lat"] = (
    charging_df["lat"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

charging_df["lon"] = (
    charging_df["lon"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

charging_df["power_kw"] = (
    charging_df["power_kw"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)


 #✅ NUR Schnelllader behalten
charging_df = charging_df[
    charging_df["power_kw"] >= 150
]

print(f"Schnellladesäulen (>=150 kW): {len(charging_df)}")


# ============================================================
# 5. MAPPING
# ============================================================

charging_nodes = []

for _, row in charging_df.iterrows():
    node_id, dist = nearest_node(row["lat"], row["lon"])

    charging_nodes.append({
        "id": row["id"],
        "node": node_id,
        "power_kw": row["power_kw"],
        "distance_m": dist
    })

charging_nodes_df = pd.DataFrame(charging_nodes)

print("\nMapping Beispiel:")
print(charging_nodes_df.head())

# ============================================================
# 6. FILTER (SEHR WICHTIG!)
# ============================================================

charging_nodes_df = charging_nodes_df[
    charging_nodes_df["distance_m"] <= 150
]

print(f"Nach Filter: {len(charging_nodes_df)}")

# ============================================================
# 7. POWER DICT
# ============================================================

charge_dict = {}

for _, row in charging_nodes_df.iterrows():
    node = row["node"]
    power = row["power_kw"]

    if node not in charge_dict:
        charge_dict[node] = power
    else:
        charge_dict[node] = max(charge_dict[node], power)

# ============================================================
# 8. GRAPH AUFBAUEN
# ============================================================

G_road = nx.DiGraph()

for _, row in df_edges.iterrows():
    u = int(row["u"])
    v = int(row["v"])
    dist_km = float(row["distance"]) / 1000

    G_road.add_edge(u, v, distance_km=dist_km)

# ✅ Ladepunkte integrieren
for n in G_road.nodes():
    G_road.nodes[n]["charging_power_kw"] = charge_dict.get(n, 0)

# ============================================================
# 9. DEBUG
# ============================================================

charging_nodes_count = sum(
    1 for n in G_road.nodes()
    if G_road.nodes[n]["charging_power_kw"] > 0
)

print(f"\n✅ Knoten mit Ladeinfrastruktur: {charging_nodes_count}")

# ============================================================
# 10. MILP READY
# ============================================================

ChargeNode = {
    n: 1 if G_road.nodes[n]["charging_power_kw"] > 0 else 0
    for n in G_road.nodes()
}

print("\nBeispiel ChargeNode:")
for k in list(ChargeNode.keys())[:10]:
    print(k, "→", ChargeNode[k])

Keys: dict_keys(['nodes', 'arcs', 'cost', 'highway', 'name', 'coords', 'Dist', 'node_coords', 'tunnel'])
Knoten: 294,724
Kanten: 533,453


C:\Users\tspol\AppData\Local\Temp\ipykernel_1848\3033629342.py:74: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(



Spalten gefunden:
['Ladeeinrichtungs-ID', 'Betreiber', 'Anzeigename (Karte)', 'Status', 'Art der Ladeeinrichtung', 'Anzahl Ladepunkte', 'Nennleistung Ladeeinrichtung [kW]', 'Inbetriebnahmedatum', 'Straße', 'Hausnummer', 'Adresszusatz', 'Postleitzahl', 'Ort', 'Kreis/kreisfreie Stadt', 'Bundesland', 'Breitengrad', 'Längengrad', 'Standortbezeichnung', 'Informationen zum Parkraum', 'Bezahlsysteme', 'Öffnungszeiten', 'Öffnungszeiten: Wochentage', 'Öffnungszeiten: Tageszeiten', 'Steckertypen1', 'Nennleistung Stecker1', 'EVSE-ID1', 'Public Key1', 'Steckertypen2', 'Nennleistung Stecker2', 'EVSE-ID2', 'Public Key2', 'Steckertypen3', 'Nennleistung Stecker3', 'EVSE-ID3', 'Public Key3', 'Steckertypen4', 'Nennleistung Stecker4', 'EVSE-ID4', 'Public Key4', 'Steckertypen5', 'Nennleistung Stecker5', 'EVSE-ID5', 'Public Key5', 'Steckertypen6', 'Nennleistung Stecker6', 'EVSE-ID6', 'Public Key6']
Ladesäulen geladen: 109646
Schnellladesäulen (>=150 kW): 21657

Mapping Beispiel:
          id      node  po

In [18]:
import pickle

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_charging.pkl", "wb") as f:
    pickle.dump(G_road, f)

print("✅ Graph gespeichert inklusive Ladeinfrastruktur")

✅ Graph gespeichert inklusive Ladeinfrastruktur


In [19]:
charging_nodes_df.head(10)


,id,node,power_kw,distance_m
6746,1032001.0,13249883984,175.0,44.895481
6747,1031974.0,10924340688,175.0,26.681771
6748,1031975.0,10924340688,175.0,26.681771
6749,1141267.0,13341508460,300.0,21.120969
6750,1043260.0,2054664417,150.0,40.948658
6751,1147016.0,1997839388,150.0,13.943088
6752,1103832.0,12946620847,150.0,18.254907
6753,1102174.0,5727158070,250.0,21.315674
6754,1102175.0,5727158070,250.0,21.315674
6755,1102176.0,5727158070,250.0,21.315674


In [24]:
# ============================================================
# 🔌 GRAPH + LADEINFRASTRUKTUR MAPPING
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx

# ============================================================
# GRAPH LADEN
# ============================================================

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_geo_com.pkl", "rb") as f:
    berlin_graph = pickle.load(f)

df_edges = pd.DataFrame(berlin_graph["arcs"])
node_coords = berlin_graph["node_coords"]

node_ids = np.array(list(node_coords.keys()))
node_lats = np.array([node_coords[n][0] for n in node_ids])
node_lons = np.array([node_coords[n][1] for n in node_ids])

print(f"Knoten: {len(node_ids):,}")
print(f"Kanten: {len(df_edges):,}")

# ============================================================
# DISTANZEN
# ============================================================

Dist_dict = berlin_graph["Dist"]

df_edges["distance"] = [
    Dist_dict.get((row["u"], row["v"]), 0)
    for _, row in df_edges.iterrows()
]

# ============================================================
# NEAREST NODE
# ============================================================

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

def nearest_node(lat, lon):
    dists = haversine_vectorized(lat, lon, node_lats, node_lons)
    idx = np.argmin(dists)
    return int(node_ids[idx]), float(dists[idx])

# ============================================================
# CSV LADEN
# ============================================================

file_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Ladesaeulenregister_BNetzA_2026-04-22.csv"

charging_df = pd.read_csv(file_path, sep=";", encoding="latin-1", header=1)
charging_df.columns = charging_df.columns.str.strip()

charging_df = pd.read_csv(
    file_path,
    sep=";",
    encoding="latin-1",
    header=10
)

charging_df = charging_df[[
    "Ladeeinrichtungs-ID",
    "Breitengrad",
    "Längengrad",
    "Nennleistung Ladeeinrichtung [kW]"
]].dropna()

charging_df.columns = ["id", "lat", "lon", "power_kw"]

# 🔥 FIX: String → Float
charging_df["lat"] = charging_df["lat"].astype(str).str.replace(",", ".").astype(float)
charging_df["lon"] = charging_df["lon"].astype(str).str.replace(",", ".").astype(float)
charging_df["power_kw"] = charging_df["power_kw"].astype(str).str.replace(",", ".").astype(float)

# ✅ Schnelllader filtern
charging_df = charging_df[charging_df["power_kw"] >= 150]

print(f"Ladesäulen: {len(charging_df)}")

# ============================================================
# MAPPING (MIT LAT/LON!)
# ============================================================

charging_nodes = []

for _, row in charging_df.iterrows():
    node_id, dist = nearest_node(row["lat"], row["lon"])

    charging_nodes.append({
        "id": row["id"],
        "node": node_id,
        "lat": row["lat"],
        "lon": row["lon"],
        "power_kw": row["power_kw"],
        "distance_m": dist
    })

charging_nodes_df = pd.DataFrame(charging_nodes)

# ✅ Filter
charging_nodes_df = charging_nodes_df[
    charging_nodes_df["distance_m"] <= 150
]

print(f"Nach Mapping: {len(charging_nodes_df)}")

# ============================================================
# GRAPH MIT LADEPUNKTEN
# ============================================================

G_road = nx.DiGraph()

for _, row in df_edges.iterrows():
    u = int(row["u"])
    v = int(row["v"])
    dist_km = float(row["distance"]) / 1000

    G_road.add_edge(u, v, distance_km=dist_km)

# Ladepower einbauen
charge_dict = {}

for _, row in charging_nodes_df.iterrows():
    node = row["node"]
    power = row["power_kw"]

    if node not in charge_dict:
        charge_dict[node] = power
    else:
        charge_dict[node] = max(charge_dict[node], power)

for n in G_road.nodes():
    G_road.nodes[n]["charging_power_kw"] = charge_dict.get(n, 0)

print("✅ Graph mit Ladeinfrastruktur fertig")

Knoten: 294,724
Kanten: 533,453


C:\Users\tspol\AppData\Local\Temp\ipykernel_1848\999663305.py:69: DtypeWarning: Columns (0: Unnamed: 0, 1: Unnamed: 5, 2: Unnamed: 11, 3: Unnamed: 40, 4: Unnamed: 42, 5: Unnamed: 44, 6: Unnamed: 46) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(file_path, sep=";", encoding="latin-1", header=1)
C:\Users\tspol\AppData\Local\Temp\ipykernel_1848\999663305.py:72: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(


Ladesäulen: 21657
Nach Mapping: 532
✅ Graph mit Ladeinfrastruktur fertig


In [25]:
import pickle

with open(r"C:\Users\tspol\OneDrive\Desktop\berlin_graph_with_charging.pkl", "wb") as f:
    pickle.dump(G_road, f)

print("✅ Graph mit Ladeinfrastruktur gespeichert")

✅ Graph mit Ladeinfrastruktur gespeichert


In [27]:
# ============================================================
# 🗺️ KARTE MIT LADESTATIONEN
# ============================================================

import folium

# Mittelpunkt Berlin
m = folium.Map(
    location=[52.52, 13.40],
    zoom_start=12,
    tiles="CartoDB positron"
)

# ============================================================
# 🔌 LADESTATIONEN
# ============================================================

for _, row in charging_nodes_df.iterrows():

    lat = row["lat"]
    lon = row["lon"]
    power = row["power_kw"]

    # 🎨 Farbcode nach Leistung
    if power >= 300:
        color = "red"
    elif power >= 150:
        color = "orange"
    else:
        color = "blue"

    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{power:.0f} kW"
    ).add_to(m)

# ============================================================
# 🛣️ STRASSENNETZ (AUSSCHNITT!)
# ============================================================

for i, (u, v) in enumerate(G_road.edges()):

    if i > 800:   # ⚠️ wichtig: sonst langsam
        break

    coord_u = node_coords.get(u)
    coord_v = node_coords.get(v)

    if coord_u and coord_v:
        folium.PolyLine(
            locations=[coord_u, coord_v],
            color="gray",
            weight=1,
            opacity=0.5
        ).add_to(m)

# ============================================================
# 👀 DIREKT ANZEIGEN (Jupyter Notebook)
# ============================================================

m.save("ladeinfrastruktur_karte.html")